In [5]:
import pandas as pd
import numpy as np



In [6]:
df = pd.read_csv('/home/patel-krish/anaconda3/BrainyBeam/womens_clothing/dataset/womens_clothing.csv')
num_cols = ['Age', 'Rating', 'Positive Feedback Count']

summary = df[num_cols].agg(['count', 'mean', 'median', 'std', 'min', 'max', 'skew']).T
print(summary.round(3))

                          count    mean  median     std   min   max   skew
Age                      1500.0  50.283    50.0  19.093  18.0  84.0  0.058
Rating                   1500.0   4.199     5.0   1.114   1.0   5.0 -1.334
Positive Feedback Count  1500.0   2.367     2.0   2.768   0.0  20.0  1.965


In [7]:
iqr_flags = pd.DataFrame(index=df.index)

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    mask = (df[col] < lower) | (df[col] > upper)
    iqr_flags[col] = mask
    print(f"{col:25s} | Bounds=[{lower:.2f}, {upper:.2f}] | Outliers={mask.sum()} ({mask.mean()*100:.2f}%)")

Age                       | Bounds=[-14.00, 114.00] | Outliers=0 (0.00%)
Rating                    | Bounds=[2.50, 6.50] | Outliers=155 (10.33%)
Positive Feedback Count   | Bounds=[-4.50, 7.50] | Outliers=83 (5.53%)


In [8]:
from scipy.stats import zscore

z_flags = pd.DataFrame(index=df.index)

for col in num_cols:
    z_vals = np.abs(zscore(df[col]))
    mask = z_vals > 3
    z_flags[col] = mask
    print(f"{col:25s} | Max |z|={z_vals.max():.2f} | Outliers (|z|>3)={mask.sum()} ({mask.mean()*100:.2f}%)")

Age                       | Max |z|=1.77 | Outliers (|z|>3)=0 (0.00%)
Rating                    | Max |z|=2.87 | Outliers (|z|>3)=0 (0.00%)
Positive Feedback Count   | Max |z|=6.37 | Outliers (|z|>3)=27 (1.80%)


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

X_scaled = StandardScaler().fit_transform(df[num_cols])
db = DBSCAN(eps=0.8, min_samples=10).fit(X_scaled)

db_outliers = (db.labels_ == -1)
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)

print(f"Clusters found: {n_clusters} | Noise/Outliers: {db_outliers.sum()} ({db_outliers.mean()*100:.2f}%)")
print("\nProfile of DBSCAN Outliers:")
print(df.loc[db_outliers, num_cols].describe().round(2))

Clusters found: 5 | Noise/Outliers: 31 (2.07%)

Profile of DBSCAN Outliers:
         Age  Rating  Positive Feedback Count
count  31.00   31.00                    31.00
mean   50.68    3.10                    11.74
std    18.65    1.37                     4.00
min    18.00    1.00                     5.00
25%    37.00    2.00                     8.50
50%    47.00    3.00                    12.00
75%    64.00    4.00                    14.00
max    83.00    5.00                    20.00


In [ ]:
df_treated = df.copy()

df_treated['Positive Feedback Count_log1p'] = np.log1p(df_treated['Positive Feedback Count'])

p99 = df_treated['Positive Feedback Count'].quantile(0.99)
df_treated['Positive Feedback Count_capped'] = df_treated['Positive Feedback Count'].clip(upper=p99)

df_treated['is_multivariate_outlier'] = db_outliers

print(f"Raw Skewness:      {df['Positive Feedback Count'].skew():.3f}")
print(f"log1p Skewness:    {df_treated['Positive Feedback Count_log1p'].skew():.3f}")
print(f"99th-pct Cap ({p99:.0f}): {df_treated['Positive Feedback Count_capped'].skew():.3f} "
      f"({(df['Positive Feedback Count'] > p99).sum()} values capped)")

Raw Skewness:      1.965
log1p Skewness:    0.227
99th-pct Cap (12): 1.529 (14 values capped)


In [11]:
df_treated.to_csv('womens_clothing_outliers_treated.csv', index=False)
print("Saved outlier-treated dataset to 'womens_clothing_outliers_treated.csv' with shape:", df_treated.shape)

Saved outlier-treated dataset to 'womens_clothing_outliers_treated.csv' with shape: (1500, 13)
